# Edge vision adaptation test: the demo video

Makes a one-minute captioned MP4 from the phone clips and the per-frame predictions saved by the main notebook (`edge_vision_adaptation_test.ipynb`). No model runs here, so a CPU runtime is enough.

1. **Load and unload cycle:** the new part (a whisk) in daylight, which nobody trained on. Every approach's verdict on every frame, and the timeline underneath.
2. **A new station:** the fast classifier trained on station 1, before and after 5 labelled frames from the new station.
3. **The speed result:** a VLM that decides vs one that writes.

**Run:** Runtime > Run all. It needs the results of the main notebook: run it from a clone of this repository and it reads `results/run4`, otherwise upload `edge_vision_results_real.zip` when asked. The 4 clips it uses download from this repository's release. About 3 minutes; the last cell shows the video and downloads it.

In [ ]:
RESULTS_DIR = "../results/run4"                 # the committed run; used when it exists
RESULTS_ZIP = "edge_vision_results_real.zip"    # otherwise the zip made by edge_vision_adaptation_test.ipynb
CLIPS_URL = "https://github.com/akamaurya/edge-vision-adaptation-test/releases/download/clips-v1/"   # the clips are attached to a release of this repository
CLIP_NAMES = ["day_cycle.MOV", "s2_empty.MOV", "s2_A.MOV", "s2_B.MOV"]
CYCLE_TRUTH_S = [(11.5, 26.5), (36.0, 50.5)]   # seconds when the whisk sits on the tray in day_cycle, read by eye from the footage
MOTION_S = 0.5                                 # frames this close to a load or unload count as "in motion": shown grey, not scored
OUT_FPS = 6                                    # output frame rate; the clips were sampled at 2 frames per second
W, H = 1280, 720
AUTHOR = "Chander Maurya Kumbkarni · September 2026"

In [ ]:
# no extra packages: numpy, pandas, matplotlib, Pillow and ffmpeg are all this needs

## 1. Inputs
The results zip, then the 4 clips, then frames at the same 2 per second the main notebook used, so frame numbers line up with its predictions.

In [ ]:
import os, json, shutil, subprocess, zipfile
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

shutil.rmtree("res", ignore_errors=True)
if os.path.isdir(RESULTS_DIR):
    shutil.copytree(RESULTS_DIR, "res")
else:
    if not os.path.exists(RESULTS_ZIP):
        from google.colab import files
        print(f"Upload {RESULTS_ZIP} from the main notebook")
        RESULTS_ZIP = next(iter(files.upload()))
    zipfile.ZipFile(RESULTS_ZIP).extractall("res")
R = json.load(open("res/results.json"))
assert not R["demo"], "these results come from DEMO mode: synthetic numbers never go in the video"
FPS, SKIP_S = R["fps_extracted"], R.get("skip_s", 10)
pred, cyc = pd.read_csv("res/predictions.csv"), pd.read_csv("res/cycle.csv")

os.makedirs("clips", exist_ok=True)
for name in CLIP_NAMES:
    if not os.path.exists(f"clips/{name}"):
        import urllib.request
        try: urllib.request.urlretrieve(CLIPS_URL + name, f"clips/{name}")
        except Exception as ex: print(f"could not download {name}: {str(ex)[:150]}")
missing = [n for n in CLIP_NAMES if not os.path.exists(f"clips/{n}") or os.path.getsize(f"clips/{n}") < 1_000_000]
if missing:
    from google.colab import files
    print("These did not download, upload them:", missing)
    for name, data in files.upload().items(): open(f"clips/{name}", "wb").write(data)
for name in CLIP_NAMES:
    clip = os.path.splitext(name)[0]
    shutil.rmtree(f"frames/{clip}", ignore_errors=True); os.makedirs(f"frames/{clip}")
    subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", f"clips/{name}", "-vf", f"fps={FPS},scale=1280:-2", f"frames/{clip}/%04d.jpg"], check=True)
print({c: len(os.listdir(f"frames/{c}")) for c in os.listdir("frames")}, "frames; cycle predictions:", len(cyc))

## 2. Drawing
Dark cards, the frame on the left, each approach's P(part) and verdict on the right. Green = right, red = wrong, grey = the part is in motion (a hand is placing or lifting it), which isn't scored.

In [ ]:
FONT = os.path.join(matplotlib.get_data_path(), "fonts/ttf/DejaVuSans.ttf")
BOLD = os.path.join(matplotlib.get_data_path(), "fonts/ttf/DejaVuSans-Bold.ttf")
font = lambda size, bold=False: ImageFont.truetype(BOLD if bold else FONT, size)
BG, PANEL, INK, MUTED = (17, 24, 39), (31, 41, 55), (243, 244, 246), (156, 163, 175)
GREEN, RED, GREY = (34, 197, 94), (239, 68, 68), (107, 114, 128)
MODEL_COLORS = {"YOLO": "#60a5fa", "Fast classifier": "#f59e0b", "Checker A": "#34d399", "Checker B": "#f472b6"}
hexrgb = lambda h: tuple(int(h[i:i + 2], 16) for i in (1, 3, 5))

shutil.rmtree("vid", ignore_errors=True); os.makedirs("vid")
n_out = 0
def emit(img, times=1):                                              # append a frame to the video, `times` times
    global n_out
    for _ in range(times):
        n_out += 1; img.save(f"vid/{n_out:05d}.jpg", quality=92)

def centred(d, y, text, size, bold=False, fill=INK):
    w = d.textlength(text, font=font(size, bold)); d.text(((W - w) / 2, y), text, font=font(size, bold), fill=fill)

def card(lines, seconds):                                            # a title card: lines of (text, size, bold, colour)
    img = Image.new("RGB", (W, H), BG); d = ImageDraw.Draw(img)
    y = (H - sum(int(s * 1.6) for _, s, _, _ in lines)) / 2
    for text, size, bold, fill in lines:
        centred(d, y, text, size, bold, fill); y += int(size * 1.6)
    emit(img, int(seconds * OUT_FPS))

def base(title, frame_path):                                         # the common layout: title bar, the camera frame on the left
    img = Image.new("RGB", (W, H), BG); d = ImageDraw.Draw(img)
    d.text((30, 18), title, font=font(24, True), fill=INK)
    img.paste(Image.open(frame_path).convert("RGB").resize((768, 432)), (30, 64))
    d.rectangle((830, 64, 1250, 496), fill=PANEL)
    d.text((30, H - 26), AUTHOR, font=font(13), fill=MUTED)
    return img, d

def pill(d, x, y, text, ok):                                         # PART / EMPTY, coloured by right, wrong or not scored
    fill = GREY if ok is None else GREEN if ok else RED
    w = d.textlength(text, font=font(17, True)) + 22
    d.rounded_rectangle((x, y, x + w, y + 30), radius=15, fill=fill)
    d.text((x + 11, y + 5), text, font=font(17, True), fill=(255, 255, 255))

def bar(d, x, y, p, colour, width=240):                              # P(part) as a bar, with a tick at the 0.5 cutoff
    d.rectangle((x, y, x + width, y + 14), fill=(55, 65, 81))
    d.rectangle((x, y, x + int(width * p), y + 14), fill=colour)
    d.line((x + width // 2, y - 4, x + width // 2, y + 18), fill=INK, width=2)

## 3. Segment 1: the load and unload cycle
The whisk goes on the tray and comes off twice. The truth comes from watching the footage (the setting `CYCLE_TRUTH_S` above); the half second around each load or unload, when a hand is moving the part, is shown grey and not scored.

In [ ]:
models = [m for m in MODEL_COLORS if m in cyc.columns]
def truth_at(t):                                                     # True / False, or None while the part is in motion
    for a, b in CYCLE_TRUTH_S:
        if a - MOTION_S <= t < a or b < t <= b + MOTION_S: return None
        if a <= t <= b: return True
    return False

# the timeline under the frame, drawn once; a cursor moves along it
fig, ax = plt.subplots(figsize=(12.2, 1.75), dpi=100)
fig.patch.set_facecolor(np.array(BG) / 255); ax.set_facecolor(np.array(BG) / 255)
for a, b in CYCLE_TRUTH_S: ax.axvspan(a, b, color="white", alpha=0.10, lw=0)
for m in models: ax.plot(cyc.t_s, cyc[m], color=MODEL_COLORS[m], lw=1.8, label=m)
ax.axhline(0.5, color="white", lw=0.8, ls="--", alpha=0.6)
ax.set_xlim(cyc.t_s.min(), cyc.t_s.max()); ax.set_ylim(-0.05, 1.05); ax.set_yticks([0, 0.5, 1])
ax.tick_params(colors="#9ca3af", labelsize=8)
for s in ax.spines.values(): s.set_color("#4b5563")
ax.set_ylabel("P(part)", color="#9ca3af", fontsize=9)
ax.legend(loc="upper left", ncol=len(models), fontsize=8, frameon=False, labelcolor="#e5e7eb", bbox_to_anchor=(0, 1.3))
ax.text(0.995, 1.2, "shaded: whisk on the tray (by eye)", transform=ax.transAxes, ha="right", color="#9ca3af", fontsize=8)
fig.subplots_adjust(left=0.05, right=0.99, bottom=0.17, top=0.8)
fig.savefig("timeline.png", facecolor=fig.get_facecolor()); plt.close(fig)
TL = Image.open("timeline.png").convert("RGB")
x0, x1 = [ax.transData.transform((x, 0))[0] for x in (cyc.t_s.min(), cyc.t_s.max())]   # pixel columns of the first and last second

right = {m: 0 for m in models}; scored = 0
def cycle_frame(row, i):
    global scored
    t, truth = row.t_s, truth_at(row.t_s)
    img, d = base("1. Load and unload cycle: the new part (whisk) in daylight, in nobody's training data", f"frames/day_cycle/{row.frame}")
    d.text((850, 76), f"t = {t:4.1f} s", font=font(16), fill=MUTED)
    d.text((970, 76), "truth: " + ("part moving" if truth is None else "PART on tray" if truth else "tray EMPTY"), font=font(16, True), fill=INK)
    if truth is not None: scored += 1
    for k, m in enumerate(models):
        y, p = 112 + 94 * k, float(row[m])
        ok = None if truth is None else (p > 0.5) == truth
        if ok: right[m] += 1
        d.text((850, y), m, font=font(19, True), fill=hexrgb(MODEL_COLORS[m]))
        d.text((850, y + 28), f"P(part) {p:.2f}", font=font(15), fill=INK)
        bar(d, 850, y + 54, p, hexrgb(MODEL_COLORS[m]))
        pill(d, 1112, y + 10, "PART" if p > 0.5 else "EMPTY", ok)
        d.text((1112, y + 48), f"right {right[m]}/{scored}", font=font(13), fill=MUTED)
    img.paste(TL, (30, 506))
    cx = int(30 + x0 + (x1 - x0) * (t - cyc.t_s.min()) / max(1e-9, cyc.t_s.max() - cyc.t_s.min()))
    d.line((cx, 512, cx, 506 + TL.height - 26), fill=(255, 255, 255), width=2)
    return img

card([("Edge vision adaptation test", 46, True, INK), ("What each approach sees, frame by frame", 26, False, INK), ("", 10, False, INK),
      ("A mock workstation filmed on a phone: a steel tray, a padlock (old part), a whisk (new part)", 18, False, MUTED),
      ("Models ran on a Colab T4 on " + R.get("run_utc", "the day of the run"), 18, False, MUTED), (AUTHOR, 16, False, MUTED)], 3.5)
card([("1. Load and unload cycle", 38, True, INK), ("The whisk goes on the tray twice, in daylight. Nobody trained on it in daylight.", 22, False, INK),
      ("Green = right, red = wrong, grey = the part is moving (not scored).", 20, False, MUTED)], 3)
for i, row in cyc.iterrows():
    img = cycle_frame(row, i)
    emit(img)
emit(img, int(1.5 * OUT_FPS))
print("segment 1 right on scored frames:", {m: f"{right[m]}/{scored}" for m in models})

## 4. Segment 2: a new station, before and after 5 labelled frames
Station 2 has a different table, tray and background. First the fast classifier trained on station 1 only, then the same classifier refit on 5 labelled frames from station 2 (one of the notebook's 5-label refits). Both are scored on the same held-out station 2 test frames.

In [ ]:
s2 = pred[(pred.station == "s2") & (pred.pool == "test")].copy()
s2["order"] = s2["clip"].map({"s2_empty": 0, "s2_A": 1, "s2_B": 2}); s2 = s2.sort_values(["order", "frame"])
curve = R["learning_curves"]["new layout"]["Fast classifier, human labels"]
refit_s = next(p["s"] for p in curve if p["n"] == 5)
checks = {k: v[0]["mean"] for k, v in R["learning_curves"]["new layout"].items() if k.startswith("Checker") and v}
what = {"s2_empty": "empty tray", "s2_A": "padlock (part A)", "s2_B": "whisk (part B)"}

def s2_phase(col, label, sub, colour):
    marks = []
    for row in s2.itertuples():
        p, truth = float(getattr(row, col)), bool(row.part)
        marks.append((p > 0.5) == truth)
        img, d = base("2. A new station, no labels of its own", f"frames/{row.clip}/{row.frame}")
        d.text((850, 76), label, font=font(30, True), fill=colour)
        d.multiline_text((850, 118), sub, font=font(15), fill=INK, spacing=4)
        d.text((850, 186), "on the tray: " + what[row.clip], font=font(16, True), fill=INK)
        d.text((850, 222), "Fast classifier", font=font(19, True), fill=hexrgb(MODEL_COLORS["Fast classifier"]))
        d.text((850, 250), f"P(part) {p:.2f}", font=font(15), fill=INK)
        bar(d, 850, 276, p, hexrgb(MODEL_COLORS["Fast classifier"]))
        pill(d, 1112, 232, "PART" if p > 0.5 else "EMPTY", marks[-1])
        d.text((850, 318), f"right so far: {sum(marks)} of {len(marks)}", font=font(20, True), fill=INK)
        d.text((850, 440), "Checkers on these frames, no training: " + ", ".join(f"{k[-1]} {v:.0%}" for k, v in checks.items()), font=font(13), fill=MUTED)
        for j in range(len(s2)):                                     # one square per test frame, filled in as it is shown
            x, y = 30 + j * (1220 / len(s2)), 530
            fill = (GREEN if marks[j] else RED) if j < len(marks) else (55, 65, 81)
            d.rectangle((x + 2, y, x + 1220 / len(s2) - 2, y + 40), fill=fill)
        for c, xs in s2.reset_index().groupby("clip").groups.items():
            d.text((30 + min(xs) * (1220 / len(s2)) + 4, 578), what[c], font=font(14), fill=MUTED)
        d.text((30, 612), f"{label}: {sum(marks)} of {len(marks)} test frames right so far", font=font(18, True), fill=colour)
        emit(img, 2)
    emit(img, int(1.5 * OUT_FPS))
    return sum(marks) / len(marks)

card([("2. A new station", 38, True, INK), ("Different table, tray and background.", 22, False, INK),
      ("The fast classifier starts from station 1, then gets 5 labelled frames from here.", 20, False, MUTED)], 3)
before = s2_phase("head_s1_p", "BEFORE", "trained on station 1 only,\nno labels from this station", (245, 158, 11))
after = s2_phase("s2_after5_p", "AFTER 5 LABELS", f"refit on 5 labelled frames from this station,\nin {refit_s * 1000:.0f} ms", GREEN)
print(f"segment 2: before {before:.0%}, after {after:.0%}")

## 5. The speed card, then encode
Numbers come straight from `results.json`. The Orin figures are estimates (T4 time x 2.5), not measurements.

In [ ]:
ms = {b["approach"]: b for b in R["budget"]}
dec, talk = ms["Checker A decides, 1 question"], ms["Checker A writes a 60-token JSON"]
gated = next((b for b in R["budget"] if "gated" in b["approach"] and "Checker B" in b["approach"]), None) or next(b for b in R["budget"] if "gated" in b["approach"])
share = ms["Fast classifier"]["orin_gpu_share_pct"] + gated["orin_gpu_share_pct"]
full = ms.get("Checker B decides, 6 questions in one request") if "Checker B" in gated["approach"] else dec
share10 = round(ms["Fast classifier"]["orin_gpu_share_pct"] + 0.1 * full["orin_gpu_share_pct"])   # if the gate sends 1 frame in 10
card([("Decide, don't write", 42, True, INK),
      (f"Checker A on a Colab T4: {dec['t4_ms']:.0f} ms per frame to decide, {talk['t4_ms']:.0f} ms to write a JSON answer", 22, False, INK),
      ("Jetson AGX Orin 32GB, estimated: fast classifier on every frame + gated checker", 22, False, INK),
      (f"about {share}% of the GPU in this test, about {share10}% if the gate sends 1 frame in 10", 22, False, INK),
      ("", 10, False, INK), ("Orin figures are estimates from the T4 (x 2.5), not measurements.", 18, False, MUTED),
      ("Notebook, clips and case study: linked with this video.", 18, False, MUTED)], 5)

subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-framerate", str(OUT_FPS), "-i", "vid/%05d.jpg",
                "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "20", "-movflags", "+faststart", "edge_vision_demo.mp4"], check=True)
print(f"edge_vision_demo.mp4: {n_out / OUT_FPS:.0f} s, {os.path.getsize('edge_vision_demo.mp4') / 1e6:.1f} MB")
try:
    from IPython.display import Video, display
    display(Video("edge_vision_demo.mp4", embed=True, width=800))
    from google.colab import files; files.download("edge_vision_demo.mp4")
except Exception:
    pass